In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    # Source check first: no 404/NotFound component exists anywhere in Frontend/src,
    # and App.jsx renders <Login /> for anonymous users on ANY path (except /signup).
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    time.sleep(1)

    driver.get("http://localhost:5173/this-route-does-not-exist-987654")
    time.sleep(3)
    body = driver.find_element(By.TAG_NAME, "body").text
    assert len(body.strip()) > 0, "Blank page with no content at all."
    assert "Welcome back" in body and driver.find_elements(By.ID, "username"), \
        f"Unexpected result for invalid URL: {body[:200]}"
    print("Anonymous invalid URL shows: Login page (no dedicated 404 UI in current frontend).")

    # Same invalid URL while authenticated falls back to the staff app (App.jsx ignores path)
    driver.get("http://localhost:5173/login")
    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    driver.get("http://localhost:5173/this-route-does-not-exist-987654")
    time.sleep(3)
    wait.until(EC.visibility_of_element_located((By.XPATH, "//nav[@aria-label='Staff']")))
    print("Authenticated invalid URL shows: staff app fallback (no dedicated 404 UI).")
    print("PASS: Invalid URL handling verified")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("50_invalid_url_FAIL.png")
finally:
    driver.quit()